In [25]:
# Construye el modelo
import pyomo.environ as pe

# Resuelve el modelo
import pyomo.opt as po

In [26]:
model = pe.ConcreteModel()

Sets: 
players enumera los nueve candidatos y positions las tres posiciones posibles.

In [27]:
model.players = pe.Set(initialize = list(range(1,10)))
model.positions = pe.Set(initialize = ["G","F","C"])



In [28]:
positions_player = {
    1: ["C"],
    2: ["G"],
    3: ["F", "C"],
    4: ["G", "F"],
    5: ["F", "C"],
    6: ["G", "F"],
    7: ["F", "C"],
    8: ["C"],
    9: ["F"],
}

In [29]:
handle_dict  = {1: 2, 2: 3, 3: 2, 4: 1, 5: 1, 6: 3, 7: 3, 8: 2, 9: 3}
shot_dict    = {1: 1, 2: 3, 3: 3, 4: 3, 5: 3, 6: 1, 7: 2, 8: 1, 9: 3}
rebound_dict = {1: 3, 2: 1, 3: 2, 4: 3, 5: 1, 6: 2, 7: 2, 8: 3, 9: 1}
defense_dict = {1: 3, 2: 2, 3: 2, 4: 1, 5: 2, 6: 3, 7: 1, 8: 2, 9: 3}

In [30]:
can_play_dict = {
    (i, p): int(p in positions_player[i])
    for i in model.players for p in model.positions
}
position_need_dict = {"G": 2, "F": 2, "C": 1}

Incializo los parámetros

In [31]:

model.handle = pe.Param(model.players, initialize=handle_dict)
model.shot = pe.Param(model.players, initialize=shot_dict)
model.rebound = pe.Param(model.players, initialize=rebound_dict)
model.defense = pe.Param(model.players, initialize=defense_dict)
model.can_play = pe.Param(model.players, model.positions, initialize=can_play_dict)
model.position_need = pe.Param(model.positions, initialize=position_need_dict)
model.team_size = pe.Param(initialize=5)
model.min_average = pe.Param(initialize=2)

Creo las variables

In [32]:
model.selected = pe.Var(model.players, within=pe.Binary)
model.assigned = pe.Var(model.players, model.positions, within= pe.Binary)

Funcion objetivo

In [33]:
def max_defense_rule(model):
    sum = 0
    for i in model.players:
        sum += model.defense[i]*model.selected[i]
    return sum

model.total_defense = pe.Objective(rule = max_defense_rule, sense = pe.maximize)

Constraints

Max jugadores

In [34]:
def team_size_rule(model):
    return sum(model.selected[i] for i in model.players) == model.team_size

model.team_size_constraint = pe.Constraint(rule = team_size_rule)

Cada jugador una posición

In [35]:
def one_position_rule(model, i):
    return sum(model.assigned[i, p] for p in model.positions) == model.selected[i]

model.one_position_constraint = pe.Constraint(model.players, rule = one_position_rule)

jugadores solo en posiciones admitidas

In [36]:
def eligibility_rule(model, i, p):
    return model.assigned[i, p] <= model.can_play[i, p]

model.eligibility_constraint = pe.Constraint(model.players, model.positions, rule=eligibility_rule)

2 G, 2 F, 1 C

In [37]:
def position_count_rule(model,p):
    return sum(model.assigned[i,p] for i in model.players )==model.position_need[p]

model.position_count_constraint = pe.Constraint(model.positions, rule = position_count_rule)

Medias mayores que 2

In [38]:

def handle_average_rule(model):
    return sum(model.handle[i] * model.selected[i] for i in model.players) >= model.min_average * model.team_size

model.handle_average_constraint = pe.Constraint(rule=handle_average_rule)

def shot_average_rule(model):
    return sum(model.shot[i] * model.selected[i] for i in model.players) >= model.min_average * model.team_size

model.shot_average_constraint = pe.Constraint(rule=shot_average_rule)

def rebound_average_rule(model):
    return sum(model.rebound[i] * model.selected[i] for i in model.players) >= model.min_average * model.team_size

model.rebound_average_constraint = pe.Constraint(rule=rebound_average_rule)


Proposiciones lógicas

In [41]:
expr = model.selected[3] + model.selected[6] <= 1
model.player_3_not_6 = pe.Constraint(expr = expr)

model.player_1_requires_4_or_5 = pe.Constraint(expr=model.selected[4] + model.selected[5] >= model.selected[1])

model.player_1_forbids_both_4_and_5 = pe.Constraint(expr=model.selected[4] + model.selected[5] <= 2 - model.selected[1])

model.player_8_xor_9 = pe.Constraint(expr=model.selected[8] + model.selected[9] == 1)



(type=<class 'pyomo.core.base.constraint.ScalarConstraint'>) on block unknown
with a new Component (type=<class
'pyomo.core.base.constraint.AbstractScalarConstraint'>). This is usually
indicative of a modelling error. To avoid this warning, use
block.del_component() and block.add_component().


Resolvemos

In [43]:

solver = po.SolverFactory("gurobi_direct")
results = solver.solve(model, tee=True)
print("Status:", results.solver.status)
print("Termination:", results.solver.termination_condition)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 47 rows, 36 columns and 136 nonzeros (Max)
Model fingerprint: 0x4cfe6eec
Model has 9 linear objective coefficients
Variable types: 0 continuous, 36 integer (36 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+00, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+01]

Found heuristic solution: objective 8.0000000
Presolve removed 47 rows and 36 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 20 available processors)

Solution count 2: 12 8 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.200000000000e+01, best bound 1.200000000000e+01

In [44]:

selected_players = [i for i in model.players if pe.value(model.selected[i]) > 0.5]
print("Defensa total:", pe.value(model.total_defense))
print("Jugadores seleccionados:", selected_players)

position_name = {"G": "Guard", "F": "Forward", "C": "Center"}
for i in selected_players:
    position = next(p for p in model.positions if pe.value(model.assigned[i, p]) > 0.5)
    print(f"Jugador {i}: {position_name[position]}")

for name, score in [("Manejo", model.handle), ("Tiro", model.shot), ("Rebote", model.rebound)]:
    average = sum(pe.value(score[i]) for i in selected_players) / pe.value(model.team_size)
    print(f"Media de {name.lower()}: {average:.2f}")


Defensa total: 12.0
Jugadores seleccionados: [1, 2, 4, 6, 9]
Jugador 1: Center
Jugador 2: Guard
Jugador 4: Forward
Jugador 6: Guard
Jugador 9: Forward
Media de manejo: 2.40
Media de tiro: 2.20
Media de rebote: 2.00
